# Preprocessing — Phase 2

**Instructions:**
1. Select the **paths** (EDF folder, `quality_summary.tsv`, JSON config, output folder)
2. Adjust the **preprocessing parameters** and **rejection thresholds**
3. Click **Load participants** to list participants and their channels
4. For each participant: adjust the **channels** to include, then check the participants to process
5. Click **Run preprocessing**

**Output structure:**

```
{output_folder}/
  reports_preprocessing/
    {id}_preprocessing_report.html    ← HTML report with artifact heatmap
    {id}_epoch_rejection.tsv          ← 1 row/epoch: file_id, epoch_idx, stage, reject_flag, flag per method
    {id}_rejection_summary.tsv        ← rejection statistics per stage and per method (per participant)
    global_epoch_rejection.tsv        ← concatenation of all participants' epoch_rejection files
    global_rejection_by_stage.tsv     ← pivot table: 1 row/stage, n and % rejected total + per method
  derivatives/
    [group sub-folders mirroring the EDF folder structure, if any]
    {id}_all-epo.fif                  ← all epochs + rejection metadata (MNE)
```

**Note:** Epochs are NOT removed here. Phase 2b will allow inspection of rejected epochs and validation/correction of the mask before saving a final `_clean-epo.fif`.

In [ ]:
import os
import io
import json
import time
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import mne
import yasa

import ipywidgets as widgets
from IPython.display import display, HTML
from ipyfilechooser import FileChooser
from specparam import SpectralModel

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')


In [ ]:
def drop_suffix_duplicates(raw):
    """Keep only the -0 variant when MNE creates -0/-1 duplicates for repeated channel names."""
    groups = {}
    for ch in raw.ch_names:
        if ch.endswith('-0') or ch.endswith('-1'):
            base = ch.rsplit('-', 1)[0]
            groups.setdefault(base, []).append(ch)
    to_drop = []
    for base, ch_list in groups.items():
        if any(c.endswith('-0') for c in ch_list):
            to_drop.extend(c for c in ch_list if not c.endswith('-0'))
    if to_drop:
        raw.drop_channels(to_drop)
    return raw, to_drop


def adapt_remap_dict_to_suffixes(raw, remap_dict):
    """Handle MNE's -0 suffix when the remap config uses the base channel name."""
    ch_set = set(raw.ch_names)
    new_remap = {}
    for base_label, target in remap_dict.items():
        if base_label in ch_set:
            new_remap[base_label] = target
        elif f'{base_label}-0' in ch_set:
            new_remap[f'{base_label}-0'] = target
    return new_remap


# ---- Rejection-method registry (single source of truth) ----
# 'event' is the optional 6th method (epochs containing selected scored events). The helpers
# below iterate METHOD_ORDER so 'event' is handled when present and silently ignored when absent.
METHOD_ORDER  = ['amplitude', 'flat', 'gradient', '1f_error', '1f_r2', 'event']
METHOD_CODE   = {m: i + 1 for i, m in enumerate(METHOD_ORDER)}   # 1..6  (0 = none)
MULTIPLE_CODE = len(METHOD_ORDER) + 1                            # 7 = multiple


# --- Custom (non-AASM) sleep stages -------------------------------------------
# Declared in config_param/custom_stages.json (written by 3_remap_hypno) and recognised here
# so per-stage rejection thresholds, summaries and the heatmap include them.
BASE_STAGE_COLORS = {'W': '#969696', 'N1': '#9e9ac8', 'N2': '#807dba', 'N3': '#6a51a3', 'R': '#c994c7'}
# Distinct, non-red palette for custom stages (red is reserved for REM on the hypnogram).
CUSTOM_STAGE_PALETTE = ['#8dd3c7', '#ffffb3', '#bebada', '#80b1d3', '#fdb462', '#b3de69', '#fccde5', '#d9d9d9']


def load_custom_stages(folder):
    """Read config_param/custom_stages.json; return list of kept non-AASM labels ([] if absent/unreadable)."""
    if not folder:
        return []
    path = Path(folder) / 'config_param' / 'custom_stages.json'
    if not path.exists():
        return []
    try:
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        stages = data.get('custom_stages', []) if isinstance(data, dict) else []
        return [str(s) for s in stages]
    except Exception:
        return []


def parse_custom_field(text):
    """Parse the comma-separated 'Custom stages' field into a clean, de-duplicated list."""
    seen = []
    for tok in str(text).split(','):
        tok = tok.strip()
        if tok and tok not in seen:
            seen.append(tok)
    return seen


def custom_stage_style(custom_stages):
    """Stage -> y-position, stage -> colour, and axis ticks for AASM + custom stages.
    AASM keeps YASA heights (W top ... N3 bottom); custom stages stack below N3 (-1, -2, ...).
    Returns (stage_y, stage_colors, ytick_pos, ytick_labels)."""
    stage_y = {'W': 4, 'R': 3, 'N1': 2, 'N2': 1, 'N3': 0}
    stage_colors = dict(BASE_STAGE_COLORS)
    for i, cs in enumerate(custom_stages):
        stage_y[cs] = -1 - i
        stage_colors[cs] = CUSTOM_STAGE_PALETTE[i % len(CUSTOM_STAGE_PALETTE)]
    ordered = sorted(stage_y.items(), key=lambda kv: kv[1])
    ytick_pos = [y for _, y in ordered]
    ytick_labels = ['REM' if s == 'R' else s for s, _ in ordered]
    return stage_y, stage_colors, ytick_pos, ytick_labels


# ---- Event companion loading: CSV first, *.edf.XML (<ScoredEvents>) fallback ----
# Same loader as 4_remap_events_edf / 7_live_explore; returns a DataFrame (Name/Start/Duration).
def event_companion_paths(edf_path, csv_suffix='_event_xml.csv'):
    """Return (csv_path_or_None, xml_path_or_None) for an EDF stem."""
    edf_path = Path(edf_path)
    csv = edf_path.with_name(f'{edf_path.stem}{csv_suffix}')
    csv = csv if csv.exists() else None
    xml = None
    for cand in (f'{edf_path.name}.XML', f'{edf_path.name}.xml'):
        p = edf_path.with_name(cand)
        if p.exists():
            xml = p
            break
    return csv, xml


def _events_df_from_csv(csv_path):
    """Parse a Compumedics event CSV -> DataFrame Name/Start/Duration, or None."""
    df = pd.read_csv(str(csv_path))
    cols = {c.lower(): c for c in df.columns}
    if 'name' in cols and 'start' in cols and 'duration' in cols:
        return df.rename(columns={cols['name']: 'Name', cols['start']: 'Start',
                                  cols['duration']: 'Duration'})[['Name', 'Start', 'Duration']]
    return None


def _events_df_from_xml(xml_path):
    """Parse the <ScoredEvents> of a Profusion CMPStudyConfig .edf.XML
    -> DataFrame Name/Start/Duration (seconds), or None."""
    root = ET.parse(str(xml_path)).getroot()
    rows = []
    for se in root.iter('ScoredEvent'):
        name_el = se.find('Name')
        if name_el is None or name_el.text is None:
            continue
        start_el = se.find('Start')
        dur_el = se.find('Duration')
        start = float(start_el.text) if (start_el is not None and start_el.text) else np.nan
        dur = float(dur_el.text) if (dur_el is not None and dur_el.text) else np.nan
        rows.append((name_el.text.strip(), start, dur))
    if not rows:
        return None
    return pd.DataFrame(rows, columns=['Name', 'Start', 'Duration'])


def load_events(edf_path, csv_suffix='_event_xml.csv'):
    """Load scored events next to the EDF, CSV-first then *.edf.XML fallback.
    Returns (DataFrame Name/Start/Duration in seconds, source) with source in
    {'csv', 'xml'}, or (None, None) when no usable event companion is found."""
    csv, xml = event_companion_paths(edf_path, csv_suffix)
    if csv is not None:
        try:
            df = _events_df_from_csv(csv)
            if df is not None:
                return df, 'csv'
        except Exception:
            pass  # fall through to the XML
    if xml is not None:
        try:
            df = _events_df_from_xml(xml)
            if df is not None:
                return df, 'xml'
        except Exception:
            pass
    return None, None


def load_event_remap(path):
    """Lenient loader for event_remap.json (strict parse, then repair one trailing comma
    before a closing } or ]). Returns {raw_label: canonical_label_or_None}."""
    with open(path, encoding='utf-8') as f:
        text = f.read()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        import re
        return json.loads(re.sub(r',(\s*[}\]])', r'\1', text))


def compute_event_epoch_mask(events_df, event_remap, selected_types, n_epochs,
                             epoch_sec=30.0):
    """Flag each 30 s epoch that CONTAINS the onset of any selected canonical event type.

    events_df      : DataFrame Name/Start/Duration (seconds) from load_events, or None
    event_remap    : {raw_label: canonical_label_or_None} from event_remap.json
    selected_types : set/list of canonical labels chosen by the user
    Returns a (n_epochs,) bool array (epoch-level; replicated across channels later).
    Containment rule: an epoch is flagged when an event's ONSET (Start) falls inside it.
    The annotated Duration is intentionally IGNORED -- clinicians often score only the
    event onset, not a precise duration, so the duration field is not trusted to widen
    the rejection. Each matching event therefore flags exactly one 30 s epoch.
    """
    mask = np.zeros(n_epochs, dtype=bool)
    selected = set(selected_types)
    if events_df is None or not selected:
        return mask
    for _, ev in events_df.iterrows():
        canonical = event_remap.get(str(ev['Name']))
        if canonical is None or canonical not in selected:
            continue
        start = float(ev['Start'])
        if not np.isfinite(start) or start < 0:
            continue
        e = int(np.floor(start / epoch_sec))
        if 0 <= e < n_epochs:
            mask[e] = True
    return mask


def compute_rejection_masks(epochs_data_uV, sfreq, hypno_epochs, ptp_thresholds,
                             flat_thresh, grad_thresh,
                             psd_freqs, psds_data_uV2, thresh_error, thresh_r2,
                             event_epoch_mask=None):
    """
    Compute per-(epoch, channel) boolean rejection masks for each method.

    Parameters
    ----------
    epochs_data_uV  : (n_epochs, n_channels, n_times) array, in µV
    sfreq           : float
    hypno_epochs    : (n_epochs,) array of str  e.g. ['W', 'N2', ...]
    ptp_thresholds  : dict  e.g. {'W': 250, 'N1': 250, 'N2': 300, 'N3': 400, 'R': 250}
    flat_thresh     : float µV  — flag epoch×channel if ptp < this value
    grad_thresh     : float µV/sample  — flag if max |diff| > this
    psd_freqs       : (n_freqs,) array or None  — full PSD freq axis (from 0.5 Hz)
    psds_data_uV2   : (n_epochs, n_channels, n_freqs) array in µV²/Hz, or None
    thresh_error    : float  — flag if specparam MAE > this
    thresh_r2       : float  — flag if specparam R² < this

    Returns
    -------
    dict of (n_epochs, n_channels) bool arrays:
        keys: 'amplitude', 'flat', 'gradient', '1f_error', '1f_r2'
    """
    n_epochs, n_channels, _ = epochs_data_uV.shape

    # Peak-to-peak amplitude vs per-stage threshold
    ptp = np.ptp(epochs_data_uV, axis=-1)  # (n_epochs, n_channels)
    ptp_thresh_arr = np.array([
        ptp_thresholds.get(s, 250.0) for s in hypno_epochs
    ])[:, np.newaxis]
    mask_amplitude = ptp > ptp_thresh_arr

    # Flat signal: ptp below flat threshold
    mask_flat = ptp < flat_thresh

    # Gradient: maximum sample-to-sample absolute difference
    grad = np.max(np.abs(np.diff(epochs_data_uV, axis=-1)), axis=-1)  # (n_epochs, n_channels)
    mask_gradient = grad > grad_thresh

    # 1/f fit quality via specparam.
    # PSD was computed from 0.5 Hz; fit is restricted to >=2 Hz to minimise low-frequency
    # artifact influence (slow waves inflate delta band in deep sleep).
    # A full peak model is used (not max_n_peaks=0) so that periodic components (spindles,
    # alpha, etc.) are removed before evaluating the aperiodic fit quality — using
    # max_n_peaks=0 forces all peak power into the aperiodic fit, degrading R2 and causing
    # almost all N2/REM epochs to be rejected.
    mask_1f_error = np.zeros((n_epochs, n_channels), dtype=bool)
    mask_1f_r2    = np.zeros((n_epochs, n_channels), dtype=bool)

    if psds_data_uV2 is not None and psd_freqs is not None:
        freq_mask = psd_freqs >= 2
        freqs_reduced = psd_freqs[freq_mask]
        sp_model = SpectralModel(
            peak_width_limits=[0.5, 20],
            aperiodic_mode='fixed',
            min_peak_height=0.3,
        )
        for ei in range(n_epochs):
            for ci in range(n_channels):
                psd_reduced = psds_data_uV2[ei, ci, freq_mask]
                try:
                    sp_model.fit(freqs_reduced, psd_reduced)
                    error = sp_model.get_metrics('error', 'mae')
                    r2    = sp_model.get_metrics('gof', 'squared')
                    if error > thresh_error:
                        mask_1f_error[ei, ci] = True
                    if r2 < thresh_r2:
                        mask_1f_r2[ei, ci] = True
                except Exception:
                    mask_1f_error[ei, ci] = True
                    mask_1f_r2[ei, ci]    = True

    mask_dict = {
        'amplitude': mask_amplitude,
        'flat':      mask_flat,
        'gradient':  mask_gradient,
        '1f_error':  mask_1f_error,
        '1f_r2':     mask_1f_r2,
    }
    # Event rejection is epoch-level: replicate the (n_epochs,) flag across all channels so it
    # plugs into the same (n_epochs, n_channels) machinery as the other methods. Added only when
    # active, so older outputs without event rejection keep their original column set.
    if event_epoch_mask is not None:
        mask_dict['event'] = np.tile(np.asarray(event_epoch_mask, dtype=bool)[:, np.newaxis],
                                     (1, n_channels))
    return mask_dict


def build_combined_method_matrix(mask_dict):
    """
    Build an (n_epochs, n_channels) integer matrix encoding which method first
    flagged each cell.  0=none, then METHOD_CODE (1..6), MULTIPLE_CODE (7) for multi-flag cells.
    """
    methods   = [m for m in METHOD_ORDER if m in mask_dict]
    shape     = next(iter(mask_dict.values())).shape
    combined  = np.zeros(shape, dtype=int)
    for name in methods:
        combined[mask_dict[name]] = METHOD_CODE[name]
    n_flags = sum(mask_dict[name].astype(int) for name in methods)
    combined[n_flags > 1] = MULTIPLE_CODE  # overwrite with 'multiple'
    return combined


def plot_rejection_heatmap(combined_matrix, ch_names, hypno_epochs, file_id, custom_stages=()):
    """
    Plot a channels x epochs heatmap coloured by rejection method.
    Top strip: hypnogram as a YASA-style step-line (W at top, N3 at bottom).
    Gray line for all stages; REM highlighted in red.
    Returns a matplotlib Figure.
    """
    # Index = method code: 0 none, 1 amplitude, 2 flat, 3 gradient, 4 1/f error, 5 1/f R2,
    # 6 event, 7 multiple (matches METHOD_CODE / MULTIPLE_CODE). 'event' colour unused when off.
    # 'event' = magenta #e84393: distinct from the green/blue/orange/red family and well legible
    # on the dark-purple 'none' background.
    COLORS = ['#1c0a3b', '#c0392b', '#2980b9', '#e67e22', '#f1c40f', '#27ae60', '#e84393', '#7b0000']
    LABELS = ['none', 'amplitude', 'flat', 'gradient', '1/f error', '1/f R2', 'event', 'multiple']
    cmap   = mcolors.ListedColormap(COLORS)
    bounds = np.arange(-0.5, 8.5, 1)
    norm   = mcolors.BoundaryNorm(bounds, cmap.N)

    # Hypnogram heights: W at top (4), N3 at bottom (0); custom stages stacked below N3.
    STAGE_H, _stage_colors, _ytick_pos, _ytick_labels = custom_stage_style(custom_stages)
    _floor = min(_ytick_pos) - 1
    hypno_y = np.array([STAGE_H.get(s, _floor) for s in hypno_epochs])

    n_channels   = len(ch_names)
    n_epochs     = combined_matrix.shape[0]
    pct_rejected = 100.0 * combined_matrix.any(axis=1).mean()

    fig_w = max(8, min(n_epochs / 6, 20))
    fig_h = max(3, 0.45 * n_channels + 2.0)
    fig, axes = plt.subplots(
        2, 1, figsize=(fig_w, fig_h),
        gridspec_kw={'height_ratios': [1.2, n_channels], 'hspace': 0.04}
    )

    # --- Hypnogram strip ---
    ax_hyp = axes[0]
    x_step = np.arange(n_epochs + 1)
    y_step = np.append(hypno_y, hypno_y[-1])

    # Full step line in gray
    ax_hyp.step(x_step, y_step, where='post', color='#555555', linewidth=1.2)

    # REM in red; each custom stage highlighted in its own colour.
    for _s in (['R'] + list(custom_stages)):
        _col = '#c0392b' if _s == 'R' else _stage_colors.get(_s)
        for ei in range(n_epochs):
            if hypno_epochs[ei] == _s:
                ax_hyp.plot([ei, ei + 1], [STAGE_H[_s], STAGE_H[_s]],
                            color=_col, linewidth=2.5, solid_capstyle='butt')

    ax_hyp.set_yticks(_ytick_pos)
    ax_hyp.set_yticklabels(_ytick_labels, fontsize=7)
    ax_hyp.set_ylim(min(_ytick_pos) - 0.8, max(_ytick_pos) + 0.8)
    ax_hyp.set_xlim(0, n_epochs)
    ax_hyp.set_xticks([])
    for sp in ['top', 'right', 'bottom']:
        ax_hyp.spines[sp].set_visible(False)

    # --- Main heatmap ---
    ax = axes[1]
    im = ax.pcolormesh(
        np.arange(n_epochs + 1), np.arange(n_channels + 1),
        combined_matrix.T,  # (n_channels, n_epochs)
        cmap=cmap, norm=norm, rasterized=True
    )
    ax.set_yticks(np.arange(n_channels) + 0.5)
    ax.set_yticklabels(ch_names, fontsize=8)
    ax.set_xlabel('Epoch index', fontsize=9)
    ax.set_xlim(0, n_epochs)
    ax.set_ylim(0, n_channels)
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)

    cbar = fig.colorbar(im, ax=ax, orientation='vertical',
                        fraction=0.02, pad=0.01, ticks=np.arange(len(LABELS)))
    cbar.ax.set_yticklabels(LABELS, fontsize=7)

    fig.suptitle(f'{file_id} — Artefacts ({pct_rejected:.0f}% rejected)',
                 fontsize=11, y=1.01)
    fig.tight_layout()
    return fig


def build_rejection_summary(mask_dict, hypno_epochs, file_id, ch_names, custom_stages=()):
    """Per-stage, per-method rejection counts DataFrame (summary statistics)."""
    methods = [m for m in METHOD_ORDER if m in mask_dict]
    rows    = []
    for stage in ['W', 'N1', 'N2', 'N3', 'R'] + list(custom_stages):
        stage_mask = (hypno_epochs == stage)
        n_total    = int(stage_mask.sum())
        any_flag   = np.zeros(len(hypno_epochs), dtype=bool)
        for name in methods:
            any_flag |= mask_dict[name].any(axis=1)
        n_rej_any = int((any_flag & stage_mask).sum())
        rows.append({
            'file_id': file_id, 'stage': stage, 'method': 'any',
            'n_total': n_total, 'n_rejected': n_rej_any,
            'pct_rejected': round(100 * n_rej_any / n_total, 1) if n_total > 0 else float('nan'),
        })
        for name in methods:
            per_epoch = mask_dict[name].any(axis=1)
            n_rej = int((per_epoch & stage_mask).sum())
            rows.append({
                'file_id': file_id, 'stage': stage, 'method': name,
                'n_total': n_total, 'n_rejected': n_rej,
                'pct_rejected': round(100 * n_rej / n_total, 1) if n_total > 0 else float('nan'),
            })
    return pd.DataFrame(rows)


def build_epoch_rejection_log(mask_dict, hypno_epochs, file_id):
    """Per-epoch rejection log: one row per epoch with epoch-level (any-channel) method flags."""
    methods  = [m for m in METHOD_ORDER if m in mask_dict]
    n_epochs = len(hypno_epochs)
    rows = []
    for ei in range(n_epochs):
        row = {
            'file_id':     file_id,
            'epoch_idx':   ei,
            'stage':       hypno_epochs[ei],
            'reject_flag': bool(any(mask_dict[m][ei].any() for m in methods)),
        }
        for m in methods:
            col = 'flag_' + m.replace('/', '').replace(' ', '')
            row[col] = bool(mask_dict[m][ei].any())
        rows.append(row)
    return pd.DataFrame(rows)


def build_global_summary_table(all_rejection_summaries, custom_stages=()):
    """
    Build a dataset-level pivot table from per-participant rejection summaries.
    One row per sleep stage; columns: n_total, n_rejected, pct_rejected (pooled),
    then n_rej_{method} and pct_rej_{method} for each rejection method.
    """
    df_all  = pd.concat(all_rejection_summaries, ignore_index=True)
    # Methods present in the loaded summaries (so 'event' columns appear only when used and
    # mixed runs — some with, some without event rejection — never raise).
    methods_list = [m for m in METHOD_ORDER if m in set(df_all['method'])]
    rows = []
    for stage in ['W', 'N1', 'N2', 'N3', 'R'] + list(custom_stages):
        stage_data = df_all[df_all['stage'] == stage]
        any_data   = stage_data[stage_data['method'] == 'any']
        n_total    = int(any_data['n_total'].sum())
        n_rej      = int(any_data['n_rejected'].sum())
        n_parts    = int(len(any_data))
        row = {
            'stage':          stage,
            'n_participants': n_parts,
            'n_total':        n_total,
            'n_rejected':     n_rej,
            'pct_rejected':   round(100 * n_rej / n_total, 1) if n_total > 0 else float('nan'),
        }
        for m in methods_list:
            m_data  = stage_data[stage_data['method'] == m]
            n_m_rej = int(m_data['n_rejected'].sum())
            row[f'n_rej_{m}']   = n_m_rej
            row[f'pct_rej_{m}'] = round(100 * n_m_rej / n_total, 1) if n_total > 0 else float('nan')
        rows.append(row)
    return pd.DataFrame(rows)


## 1 — Paths

In [ ]:
fc_edf = FileChooser()
fc_edf.title = '<b>Select your data folder:</b>'
fc_edf.show_only_dirs = True

fc_quality = FileChooser(filter_pattern='*.tsv')
fc_quality.title = '<b>quality_summary.tsv</b> (from quality_overview) :'

fc_config = FileChooser(filter_pattern='*.json')
fc_config.title = '<b>remap_reref_persubject.json</b> (from select&amp;remap_channels_edf) :'

fc_events = FileChooser(filter_pattern='*.json')
fc_events.title = '<b>event_remap.json</b> (from remap_events) — optional, for event-based rejection :'

csv_suffix = widgets.Text(
    value='_event_xml.csv',
    description='Event CSV suffix:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)
csv_suffix_info = widgets.HTML(value='')

hypno_suffix = widgets.Text(
    value='_Hypnogram_remapped.txt',
    description='Hypno suffix:',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)
hypno_suffix_info = widgets.HTML(value='')

custom_stages_box = widgets.Text(
    value='',
    description='Custom stages:',
    placeholder='non-AASM labels to keep, e.g. N4 (auto-filled from config)',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='420px')
)
custom_stages_info = widgets.HTML(value='')


def _detect_hypno_suffixes(chooser):
    """Auto-detect hypnogram .txt suffixes in the EDF folder and populate the widget."""
    try:
        if not chooser.selected:
            hypno_suffix_info.value = ''
            csv_suffix_info.value = ''
            fc_quality.reset()
            fc_config.reset()
            fc_events.reset()
            fc_out.reset()
            return
        edf_folder = Path(chooser.selected)
        fc_quality.reset(path=str(edf_folder))
        fc_config.reset(path=str(edf_folder))
        _cp_dir = edf_folder / 'config_param'
        fc_events.reset(path=str(_cp_dir if _cp_dir.is_dir() else edf_folder))
        fc_out.reset(path=str(edf_folder))
        _cs = load_custom_stages(edf_folder)
        custom_stages_box.value = ', '.join(_cs)
        custom_stages_info.value = (
            f'<small style="color:#2e7d32;">Custom stage(s) loaded: {", ".join(_cs)}</small>' if _cs
            else '<small style="color:#888;">No custom stages registered for this dataset.</small>'
        )
        edf_files  = [f for f in sorted(edf_folder.rglob('*')) if f.suffix.lower() == '.edf' and not f.name.startswith('._')]
        if not edf_files:
            hypno_suffix_info.value = (
                '<small style="color:#888;">No EDF files found in selected folder (recursive scan)</small>'
            )
            return
        n_total = len(edf_files)
        # ---- Event CSV suffix auto-detection (most frequent suffix; shortest on ties) ----
        all_csv = [f for f in edf_folder.rglob('*') if f.suffix.lower() == '.csv']
        csv_counts = {}
        for edf in edf_files:
            for c in all_csv:
                if os.path.normcase(c.name).startswith(os.path.normcase(edf.stem)):
                    suf = c.name[len(edf.stem):]
                    csv_counts[suf] = csv_counts.get(suf, 0) + 1
        if csv_counts:
            best_csv, best_csv_n = min(csv_counts.items(), key=lambda x: (-x[1], len(x[0])))
            csv_suffix.value = best_csv
            cparts = [f'<b>{s}</b>&nbsp;(×{c})' for s, c in sorted(csv_counts.items(), key=lambda x: -x[1])]
            ccolor = '#2e7d32' if best_csv_n == n_total else '#e67e00'
            csv_suffix_info.value = (
                f'<small style="color:{ccolor};">Event CSV detected: &nbsp;{"&nbsp;·&nbsp;".join(cparts)}'
                f'&nbsp;— {best_csv_n}/{n_total} files matching</small>'
            )
        else:
            csv_suffix_info.value = (
                '<small style="color:#888;">No event CSV detected next to the EDF files — '
                'the XML fallback (*.edf.XML) will be used if present.</small>'
            )
        all_txt = [f for f in edf_folder.rglob('*') if f.suffix.lower() == '.txt']
        suffix_counts = {}
        for edf in edf_files:
            for txt in all_txt:
                if os.path.normcase(txt.name).startswith(os.path.normcase(edf.stem)):
                    suffix = txt.name[len(edf.stem):]
                    suffix_counts[suffix] = suffix_counts.get(suffix, 0) + 1
        if not suffix_counts:
            hypno_suffix_info.value = (
                '<small style="color:#e67e00;">No hypnogram .txt files detected — '
                'please verify that the files are present or adjust the suffix manually.</small>'
            )
            return
        # Among suffixes appearing for >=50% of max count, prefer the longest
        # (more specific = remapped/processed version). Tiebreaker: highest count.
        max_count = max(suffix_counts.values())
        candidates = {s: c for s, c in suffix_counts.items() if c >= max_count * 0.5}
        best_suffix, best_count = max(candidates.items(), key=lambda x: (len(x[0]), x[1]))
        hypno_suffix.value = best_suffix
        parts = [f'<b>{s}</b>&nbsp;(×{c})' for s, c in sorted(suffix_counts.items(), key=lambda x: -x[1])]
        color = '#2e7d32' if best_count == n_total else '#e67e00'
        hypno_suffix_info.value = (
            f'<small style="color:{color};">Detected: &nbsp;{"&nbsp;·&nbsp;".join(parts)}'
            f'&nbsp;— {best_count}/{n_total} files matching</small>'
        )
    except Exception as e:
        hypno_suffix_info.value = (
            f'<small style="color:#c0392b;">Error detecting hypnograms: {e}</small>'
        )
    _update_existing_reports_info()


def _update_existing_reports_info(chooser=None):
    if not fc_edf.selected or not fc_out.selected:
        existing_reports_info.value = ''
        return
    try:
        edf_folder = Path(fc_edf.selected)
        out_root = Path(fc_out.selected)
        edf_files = [f for f in sorted(edf_folder.rglob('*')) if f.suffix.lower() == '.edf' and not f.name.startswith('._')]
        n_total = len(edf_files)
        if n_total == 0:
            existing_reports_info.value = '<small style="color:#888;">No EDF files found in selected folder.</small>'
            return
        reports_dir = out_root / 'reports_preprocessing'
        n_existing = sum(
            1 for f in edf_files
            if (reports_dir / f'{f.stem}_preprocessing_report.html').exists()
        )
        existing_reports_info.value = (
            f'<small style="color:#555;">'
            f'{n_existing} / {n_total} participant(s) already have an existing preprocessing report.</small>'
        )
    except Exception as e:
        existing_reports_info.value = f'<small style="color:#c0392b;">Error checking existing reports: {e}</small>'


fc_edf.register_callback(_detect_hypno_suffixes)

existing_reports_info = widgets.HTML(value='')

skip_existing = widgets.Checkbox(
    value=True,
    description='Skip participants with an existing report',
    style={'description_width': 'initial'}
)

fc_out = FileChooser()
fc_out.title = '<b>Output folder</b> (will receive <em>reports_preprocessing/</em> and <em>derivatives/</em>) :'
fc_out.show_only_dirs = True
fc_out.register_callback(_update_existing_reports_info)


def _populate_event_types(chooser=None):
    """Rebuild one checkbox per canonical event type from event_remap.json's values."""
    if 'event_types_box' not in globals():
        return
    event_type_checkboxes.clear()
    try:
        if fc_events.selected:
            types = sorted({v for v in load_event_remap(fc_events.selected).values() if v})
        else:
            types = []
    except Exception:
        types = []
    if not types:
        event_types_box.children = [event_types_hint]
        return
    for t in types:
        event_type_checkboxes[t] = widgets.Checkbox(
            value=False, description=t, indent=False,
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='auto', margin='2px 14px 2px 0'))
    event_types_box.children = list(event_type_checkboxes.values())


fc_events.register_callback(_populate_event_types)

display(
    fc_edf,
    hypno_suffix,
    hypno_suffix_info,
    custom_stages_box,
    custom_stages_info,
    fc_config,
    fc_quality,
    fc_events,
    csv_suffix,
    csv_suffix_info,
    fc_out,
    existing_reports_info,
    skip_existing,
)

## 2 — Preprocessing and rejection parameters

In [ ]:
# ---- Resampling ----
cb_resample = widgets.Checkbox(
    value=False, description='Activate resampling',
    style={'description_width': 'initial'}
)
txt_target_freq = widgets.BoundedIntText(
    value=256, min=1, max=10000, step=1,
    description='Target frequency (Hz) :',
    style={'description_width': '180px'},
    layout=widgets.Layout(width='320px', display='none')
)

def _toggle_resample(change):
    txt_target_freq.layout.display = '' if change['new'] else 'none'

cb_resample.observe(_toggle_resample, names='value')

# ---- Filter ----
cb_filter = widgets.Checkbox(
    value=True, description='Activate bandpass filter',
    style={'description_width': 'initial'}
)
txt_l_freq = widgets.BoundedFloatText(
    value=0.1, min=0.0, max=100.0, step=0.05,
    description='l_freq (Hz) :',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='260px')
)
txt_h_freq = widgets.BoundedFloatText(
    value=40.0, min=1.0, max=500.0, step=1.0,
    description='h_freq (Hz) :',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='260px')
)

# ---- Epoch rejection thresholds ----
_ws = {'description_width': '200px'}
_wl = widgets.Layout(width='360px')

txt_thresh_W  = widgets.BoundedFloatText(value=250.0, min=1.0, max=5000.0, step=10.0,
    description='Amplitude threshold W (µV):', style=_ws, layout=_wl)
txt_thresh_N1 = widgets.BoundedFloatText(value=250.0, min=1.0, max=5000.0, step=10.0,
    description='Amplitude threshold N1 (µV):', style=_ws, layout=_wl)
txt_thresh_N2 = widgets.BoundedFloatText(value=300.0, min=1.0, max=5000.0, step=10.0,
    description='Amplitude threshold N2 (µV):', style=_ws, layout=_wl)
txt_thresh_N3 = widgets.BoundedFloatText(value=300.0, min=1.0, max=5000.0, step=10.0,
    description='Amplitude threshold N3 (µV):', style=_ws, layout=_wl)
txt_thresh_R  = widgets.BoundedFloatText(value=250.0, min=1.0, max=5000.0, step=10.0,
    description='Amplitude threshold REM (µV):', style=_ws, layout=_wl)

# Per-custom-stage amplitude thresholds (rebuilt when the 'Custom stages' field changes).
custom_thresh_widgets = {}
custom_thresh_box = widgets.VBox([])


def rebuild_custom_thresholds(*_):
    cs = parse_custom_field(custom_stages_box.value)
    rows, new_widgets = [], {}
    for s in cs:
        w = custom_thresh_widgets.get(s) or widgets.BoundedFloatText(
            value=250.0, min=1.0, max=5000.0, step=10.0,
            description=f'Amplitude threshold {s} (µV):', style=_ws, layout=_wl)
        new_widgets[s] = w
        rows.append(w)
    custom_thresh_widgets.clear()
    custom_thresh_widgets.update(new_widgets)
    custom_thresh_box.children = tuple(rows)


custom_stages_box.observe(rebuild_custom_thresholds, names='value')
rebuild_custom_thresholds()
txt_flat      = widgets.BoundedFloatText(value=1.0, min=0.01, max=100.0, step=0.1,
    description='Flat signal < (µV):', style=_ws, layout=_wl)
txt_grad      = widgets.BoundedFloatText(value=100.0, min=1.0, max=5000.0, step=10.0,
    description='Max gradient (µV/sample):', style=_ws, layout=_wl)
txt_1f_error  = widgets.BoundedFloatText(value=0.15, min=0.0, max=1.0, step=0.01,
    description='1/f error threshold >:', style=_ws, layout=_wl)
txt_1f_r2     = widgets.BoundedFloatText(value=0.85, min=0.0, max=1.0, step=0.01,
    description='1/f R² threshold <:', style=_ws, layout=_wl)


# ---- Event-based rejection ----
cb_event_reject = widgets.Checkbox(
    value=False, description='Reject epochs containing scored events',
    style={'description_width': 'initial'}
)
event_explain = widgets.HTML(
    value='<small style="color:#555;">An epoch is rejected when the <b>onset</b> of a '
          'checked event falls inside it. The annotated event <b>duration is ignored</b> '
          '(often only the onset is annotated, not the precise duration), so each event '
          'flags the single 30&nbsp;s epoch that contains its start time.</small>'
)
event_types_hint = widgets.HTML(
    value='<small style="color:#888;">Select an <b>event_remap.json</b> in Section 1 to '
          'populate the event types.</small>'
)
event_type_checkboxes = {}   # canonical label -> Checkbox (rebuilt from event_remap.json)

def selected_event_types():
    """Canonical event labels currently checked (empty list when none / unpopulated)."""
    return [lbl for lbl, cb in event_type_checkboxes.items() if cb.value]

# All event types are shown at once as a wrapping row of checkboxes (no scroll box).
event_types_box = widgets.Box([event_types_hint],
    layout=widgets.Layout(display='flex', flex_flow='row wrap', width='100%'))
btn_count_events = widgets.Button(description='Count affected epochs',
    layout=widgets.Layout(width='220px'))
out_count_events = widgets.Output()
event_box = widgets.VBox([
    event_explain, event_types_box,
    btn_count_events, out_count_events,
])

def _toggle_event_reject(change):
    event_box.layout.display = '' if change['new'] else 'none'

_toggle_event_reject({'new': cb_event_reject.value})
cb_event_reject.observe(_toggle_event_reject, names='value')


def _count_affected_epochs(btn):
    # Count, over the participants CHECKED in Section 3, how many 30 s epochs each selected
    # event type would flag (no signal is read — only the hypnogram length/stages + events).
    out_count_events.clear_output()
    with out_count_events:
        if 'participant_ui' not in globals() or not participant_ui:
            print('Load participants first (Section 3), then check the ones to count.')
            return
        checked = [fid for fid, ui in participant_ui.items() if ui['cb'].value]
        if not checked:
            print('No participant checked in Section 3.')
            return
        if not fc_events.selected:
            print('Select an event_remap.json in Section 1 first.')
            return
        types = selected_event_types()
        if not types:
            print('Select at least one event type.')
            return
        if not fc_edf.selected:
            print('Select the EDF data folder in Section 1 first.')
            return
        try:
            event_remap = load_event_remap(fc_events.selected)
        except Exception as e:
            print(f'Could not read event_remap.json: {e}')
            return
        edf_folder = Path(fc_edf.selected)
        hypno_lookup = {os.path.normcase(p.name): p for p in edf_folder.rglob('*') if p.suffix.lower() == '.txt'}
        stages = ['W', 'N1', 'N2', 'N3', 'R'] + parse_custom_field(custom_stages_box.value)
        rowtypes = types + ['(any selected)']
        counts = {t: {'n': 0, 'stage': {s: 0 for s in stages}} for t in rowtypes}
        total_epochs = 0
        stage_totals = {s: 0 for s in stages}
        n_files_ok = 0
        n_no_events = 0
        for fid in checked:
            edf_cand = [f for f in edf_folder.rglob('*')
                        if os.path.normcase(f.stem) == os.path.normcase(fid) and f.suffix.lower() == '.edf']
            if not edf_cand:
                continue
            hp = hypno_lookup.get(os.path.normcase(f'{fid}{hypno_suffix.value}'))
            if hp is None or not hp.exists():
                continue
            try:
                hyp = np.loadtxt(str(hp), dtype=str).astype('<U10')
            except Exception:
                continue
            n_ep = len(hyp)
            events_df, _src = load_events(edf_cand[0], csv_suffix.value)
            if events_df is None:
                n_no_events += 1
                continue
            n_files_ok += 1
            total_epochs += n_ep
            for s in stages:
                stage_totals[s] += int((hyp == s).sum())
            any_mask = np.zeros(n_ep, dtype=bool)
            for t in types:
                m = compute_event_epoch_mask(events_df, event_remap, [t], n_ep, 30.0)
                counts[t]['n'] += int(m.sum())
                for s in stages:
                    counts[t]['stage'][s] += int((m & (hyp == s)).sum())
                any_mask |= m
            counts['(any selected)']['n'] += int(any_mask.sum())
            for s in stages:
                counts['(any selected)']['stage'][s] += int((any_mask & (hyp == s)).sum())
        if n_files_ok == 0:
            print(f'No event companions found for the checked participants '
                  f'(checked {len(checked)}, {n_no_events} without events).')
            return
        def _pct(n, tot):
            return f'{100 * n / tot:.1f}%' if tot else '—'
        head = ''.join(f'<th style="padding:3px 10px;text-align:right;">{s}</th>' for s in stages)
        body = ''
        for t in rowtypes:
            c = counts[t]
            per_stage = ''.join(
                (f'<td style="padding:3px 10px;text-align:right;">{c["stage"][s]}'
                 f'<small style="color:#888;"> ({_pct(c["stage"][s], stage_totals[s])})</small></td>')
                if stage_totals[s] else '<td style="padding:3px 10px;text-align:right;">—</td>'
                for s in stages
            )
            weight = 'font-weight:600;' if t == '(any selected)' else ''
            body += (f'<tr style="{weight}"><td style="padding:3px 10px;">{t}</td>'
                     f'<td style="padding:3px 10px;text-align:right;">{c["n"]}</td>'
                     f'<td style="padding:3px 10px;text-align:right;">{_pct(c["n"], total_epochs)}</td>'
                     f'{per_stage}</tr>')
        html = (
            f'<p>{n_files_ok} participant(s) with events · {total_epochs} epochs total'
            + (f' · {n_no_events} without event companion' if n_no_events else '') + '</p>'
            '<table style="border-collapse:collapse;font-size:.9em;">'
            '<tr><th style="padding:3px 10px;text-align:left;">Event type</th>'
            '<th style="padding:3px 10px;text-align:right;">n epochs</th>'
            '<th style="padding:3px 10px;text-align:right;">%</th>'
            f'{head}</tr>{body}</table>'
        )
        display(HTML(html))

btn_count_events.on_click(_count_affected_epochs)


display(
    HTML('<h3 style="margin:8px 0 4px;">Preprocessing</h3>'),
    HTML('<b>Resampling</b>'),
    cb_resample, txt_target_freq,
    HTML('<br><b>Bandpass filter</b>'),
    cb_filter, widgets.HBox([txt_l_freq, txt_h_freq]),
    HTML('<hr style="margin:14px 0 4px;"><h3 style="margin:8px 0 4px;">Epoch rejection</h3>'),
    HTML('<b>Peak-to-peak amplitude per stage</b>'),
    widgets.HBox([txt_thresh_W, txt_thresh_N1]),
    widgets.HBox([txt_thresh_N2, txt_thresh_N3]),
    txt_thresh_R,
    custom_thresh_box,
    HTML('<br><b>Flat signal &amp; gradient</b>'),
    widgets.HBox([txt_flat, txt_grad]),
    HTML('<br><b>1/f fit quality (specparam)</b>'),
    widgets.HBox([txt_1f_error, txt_1f_r2]),
    HTML('<br><b>Reject epochs containing events</b>'),
    cb_event_reject, event_box,
)

## 3 — Select participants & run preprocessing

In [ ]:
participant_ui         = {}   # file_id -> {'cb': Checkbox, 'channels': {ch: Checkbox}}
participant_ui_defaults = {}   # file_id -> {'cb': bool, 'channels': {ch: bool}}
_reset_in_progress      = [False]
vbox_participants       = widgets.VBox([])

btn_load  = widgets.Button(description='Load participants',
                           button_style='info',
                           layout=widgets.Layout(width='240px', height='36px'))
load_info = widgets.HTML(value='')


def on_load_participants(btn):
    btn.disabled = True
    if not fc_quality.selected or not fc_config.selected:
        load_info.value = ('<span style="color:#c0392b;">'
                           'Please select quality_summary.tsv and the JSON config first.</span>')
        btn.disabled = False
        return
    try:
        # dtype={'file_id': str} ensures purely numeric IDs (e.g. 52, 117) are kept as
        # strings, so they match both JSON config keys (always str) and Path.stem (always str).
        df_q = pd.read_csv(fc_quality.selected, sep='\t', dtype={'file_id': str})
        with open(fc_config.selected, 'r', encoding='utf-8') as _f:
            cfg = json.load(_f)
    except Exception as e:
        load_info.value = f'<span style="color:#c0392b;">Error: {e}</span>'
        btn.disabled = False
        return

    if 'exclude' in df_q.columns:
        excl_all = df_q.groupby('file_id')['exclude'].all()
    else:
        excl_all = pd.Series(dtype=bool)

    all_ids = sorted(df_q['file_id'].unique())

    # Filter out participants with an existing report when skip_existing is checked.
    n_hidden = 0
    if skip_existing.value and fc_out.selected:
        reports_dir_check = Path(fc_out.selected) / 'reports_preprocessing'
        ids_to_load = []
        for fid in all_ids:
            if (reports_dir_check / f'{fid}_preprocessing_report.html').exists():
                n_hidden += 1
            else:
                ids_to_load.append(fid)
    else:
        ids_to_load = list(all_ids)

    participant_blocks = []
    global participant_ui, participant_ui_defaults
    participant_ui = {}
    participant_ui_defaults = {}

    # [J] Chaque participant est wrappé indépendamment pour ne pas bloquer les autres
    load_errors = []
    for fid in ids_to_load:
        try:
            fully_excluded = bool(excl_all.get(fid, False))
            not_in_cfg     = fid not in cfg

            fid_rows = df_q[df_q['file_id'] == fid]
            if 'channel' in df_q.columns and 'exclude' in df_q.columns:
                chs_flags = {row['channel']: bool(row['exclude'])
                             for _, row in fid_rows.iterrows()}
            elif 'channel' in df_q.columns:
                chs_flags = {row['channel']: False for _, row in fid_rows.iterrows()}
            else:
                chs_flags = {ch: False for ch in cfg.get(fid, {}).get('remap', {}).keys()}

            n_ch   = len(chs_flags)
            n_excl = sum(1 for excl in chs_flags.values() if excl)
            suffix_label = ''
            if fully_excluded:
                suffix_label = ' (all channels excluded)'
            elif not_in_cfg:
                suffix_label = ' (not found in JSON config)'

            p_cb = widgets.Checkbox(
                value=(not fully_excluded and not not_in_cfg),
                description=f'{fid}{suffix_label}  [{n_ch} channels, {n_excl} excluded]',
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='440px')
            )

            ch_cbs = {}
            for ch, excl in chs_flags.items():
                ch_cb = widgets.Checkbox(
                    value=not excl,
                    description=ch,
                    indent=False,
                    style={'description_width': 'initial'},
                    layout=widgets.Layout(width='120px')
                )
                ch_cbs[ch] = ch_cb

            ch_row = widgets.HBox(
                list(ch_cbs.values()),
                layout=widgets.Layout(flex_wrap='wrap', margin='0 0 10px 28px')
            )
            participant_ui[fid] = {'cb': p_cb, 'channels': ch_cbs}
            participant_ui_defaults[fid] = {
                'cb': p_cb.value,
                'channels': {ch: cb.value for ch, cb in ch_cbs.items()}
            }
            p_cb.observe(make_p_cb_observer(fid), names='value')
            participant_blocks.append(widgets.VBox([p_cb, ch_row]))
        except Exception as e:
            load_errors.append(str(fid))
            participant_blocks.append(widgets.HTML(
                value=f'<span style="color:#c0392b;">{fid} — loading error: {e}</span>'
            ))

    vbox_participants.children = participant_blocks
    n_ok = len(ids_to_load) - len(load_errors)
    msg = f'<span style="color:#2e7d32;">{n_ok} participant(s) loaded.</span>'
    if n_hidden:
        msg += (f'&nbsp;&nbsp;<span style="color:#888;">'
                f'{n_hidden} hidden (existing report).</span>')
    if load_errors:
        msg += (f'&nbsp;&nbsp;<span style="color:#c0392b;">{len(load_errors)} error(s): '
                f'{", ".join(load_errors)}</span>')
    load_info.value = msg
    btn.disabled = False


btn_load.on_click(on_load_participants)


def make_p_cb_observer(fid):
    def _on_participant_toggle(change):
        if _reset_in_progress[0]:
            return
        for ch_cb in participant_ui[fid]['channels'].values():
            ch_cb.value = change['new']
    return _on_participant_toggle


btn_select_all   = widgets.Button(description='Select all',        layout=widgets.Layout(width='120px'))
btn_deselect_all = widgets.Button(description='Deselect all',      layout=widgets.Layout(width='130px'))
btn_reset_sel    = widgets.Button(description='Reset to defaults',  layout=widgets.Layout(width='160px'))
ctrl_row         = widgets.HBox([btn_select_all, btn_deselect_all, btn_reset_sel],
                                layout=widgets.Layout(margin='4px 0 8px 0'))


def _select_all(btn):
    for fid, ui in participant_ui.items():
        ui['cb'].value = True
        for ch_cb in ui['channels'].values():
            ch_cb.value = True


def _deselect_all(btn):
    for fid, ui in participant_ui.items():
        ui['cb'].value = False
        for ch_cb in ui['channels'].values():
            ch_cb.value = False


def _reset_to_defaults(btn):
    _reset_in_progress[0] = True
    for fid, ui in participant_ui.items():
        if fid in participant_ui_defaults:
            ui['cb'].value = participant_ui_defaults[fid]['cb']
            for ch, ch_cb in ui['channels'].items():
                if ch in participant_ui_defaults[fid]['channels']:
                    ch_cb.value = participant_ui_defaults[fid]['channels'][ch]
    _reset_in_progress[0] = False


btn_select_all.on_click(_select_all)
btn_deselect_all.on_click(_deselect_all)
btn_reset_sel.on_click(_reset_to_defaults)


# ---- Run button ----
btn_run      = widgets.Button(description='▶  Run preprocessing',
                              button_style='primary',
                              layout=widgets.Layout(width='260px', height='40px'))
progress     = widgets.IntProgress(min=0, max=1, value=0, bar_style='info',
                                   layout=widgets.Layout(width='500px'))
progress_lbl = widgets.Label(value='')
out_run      = widgets.Output()


def run_preprocessing(btn):
    btn.disabled = True
    out_run.clear_output()

    if not all([fc_edf.selected, fc_quality.selected, fc_config.selected, fc_out.selected]):
        with out_run:
            print('ERROR: Please select all required paths (EDF, quality_summary, JSON config, output).')
        btn.disabled = False
        return

    edf_folder = Path(fc_edf.selected)
    out_root   = Path(fc_out.selected)

    try:
        with open(fc_config.selected, 'r', encoding='utf-8') as _f:
            config_dict = json.load(_f)
    except Exception as e:
        with out_run:
            print(f'ERROR loading config: {e}')
        btn.disabled = False
        return

    reports_dir = out_root / 'reports_preprocessing'
    try:
        reports_dir.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        with out_run:
            print(f'ERROR creating output folder: {e}')
        btn.disabled = False
        return

    do_resample      = cb_resample.value
    target_freq      = int(txt_target_freq.value) if do_resample else None
    do_filter        = cb_filter.value
    l_freq_val       = float(txt_l_freq.value)
    h_freq_val       = float(txt_h_freq.value)
    ptp_thresh = {
        'W':  float(txt_thresh_W.value),
        'N1': float(txt_thresh_N1.value),
        'N2': float(txt_thresh_N2.value),
        'N3': float(txt_thresh_N3.value),
        'R':  float(txt_thresh_R.value),
    }
    custom_stages = parse_custom_field(custom_stages_box.value)
    for _cs, _cw in custom_thresh_widgets.items():
        ptp_thresh[_cs] = float(_cw.value)
    flat_thresh_val  = float(txt_flat.value)
    grad_thresh_val  = float(txt_grad.value)
    thresh_error_val = float(txt_1f_error.value)
    thresh_r2_val    = float(txt_1f_r2.value)
    do_event         = cb_event_reject.value
    event_types      = set(selected_event_types())
    event_remap_dict = {}
    if do_event:
        if not event_types:
            with out_run:
                print('⚠ Event-based rejection is on but no event type selected — disabled for this run.')
            do_event = False
        elif not fc_events.selected:
            with out_run:
                print('⚠ Event-based rejection is on but no event_remap.json selected — disabled for this run.')
            do_event = False
        else:
            try:
                event_remap_dict = load_event_remap(fc_events.selected)
            except Exception as e:
                with out_run:
                    print(f'⚠ Could not read event_remap.json ({e}) — event-based rejection disabled for this run.')
                do_event = False

    selected_ids = [fid for fid, ui in participant_ui.items() if ui['cb'].value]
    if not selected_ids:
        with out_run:
            print('No participants selected.')
        btn.disabled = False
        return

    progress.max   = len(selected_ids)
    progress.value = 0
    all_rejection_summaries = []
    all_epoch_logs          = []
    failed                  = []
    attempted_ids           = set()
    t_start                 = time.time()
    hypno_lookup = {os.path.normcase(p.name): p for p in edf_folder.rglob('*') if p.suffix.lower() == '.txt'}

    for idx, file_id in enumerate(selected_ids):
        progress_lbl.value = f'{file_id}  ({idx + 1}/{len(selected_ids)})'

        if skip_existing.value and (reports_dir / f'{file_id}_preprocessing_report.html').exists():
            with out_run:
                print(f'[{file_id}] Skipped (preprocessing report already exists).')
            progress.value = idx + 1
            continue

        attempted_ids.add(file_id)

        try:  # [A] catch-all par participant — toute exception imprévue est capturée ici

            edf_candidates = [f for f in edf_folder.rglob('*') if os.path.normcase(f.stem) == os.path.normcase(file_id) and f.suffix.lower() == '.edf']
            if not edf_candidates:
                with out_run:
                    print(f'[{file_id}] EDF not found in {edf_folder}')
                failed.append({'file_id': file_id, 'reason': 'EDF not found'})
                progress.value += 1
                continue
            edf_path = edf_candidates[0]

            edf_rel   = edf_path.parent.relative_to(edf_folder)
            deriv_dir = out_root / 'derivatives' / edf_rel
            deriv_dir.mkdir(parents=True, exist_ok=True)

            if file_id not in config_dict:
                with out_run:
                    print(f'[{file_id}] Not found in JSON config — skipped.')
                failed.append({'file_id': file_id, 'reason': 'not in JSON config'})
                progress.value += 1
                continue

            sub_config        = config_dict[file_id]
            ch_ui             = participant_ui[file_id]['channels']
            selected_channels = [ch for ch, cb in ch_ui.items() if cb.value]
            if not selected_channels:
                with out_run:
                    print(f'[{file_id}] No channels selected — skipped.')
                failed.append({'file_id': file_id, 'reason': 'no channels selected'})
                progress.value += 1
                continue

            # [load] preload=False : on ne lit que l'en-tête pour l'instant.
            # include= utilise les noms d'ORIGINE du JSON (remap.keys()). Passer include= à la
            # LECTURE (et non un pick paresseux après coup) est important : (a) ça exclut
            # d'emblée les canaux hors-montage à fréquence plus élevée (ex. ECG 512 Hz), ce qui
            # évite un AssertionError du lecteur EDF de MNE sur la lecture partielle ET préserve
            # la fréquence native de l'EEG ; (b) ça évite tout dictionnaire de remap inversé.
            try:
                raw = mne.io.read_raw_edf(
                    str(edf_path), preload=False, encoding='latin-1',
                    include=list(sub_config.get('remap', {}).keys()), verbose=False
                )
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Error loading EDF: {e}')
                failed.append({'file_id': file_id, 'reason': f'EDF loading: {e}'})
                progress.value += 1
                continue

            raw, _ = drop_suffix_duplicates(raw)

            # [B] Renommage canaux origine -> remappé — non-fatal en soi, mais la sélection
            # qui suit en dépend (cf. garde-fou 'present' juste après).
            try:
                remap_adapted = adapt_remap_dict_to_suffixes(raw, sub_config['remap'])
                raw.rename_channels(remap_adapted)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Error renaming channels: {e} — channels not renamed.')

            # selected_channels porte les noms *remappés* (UI / quality_summary), même
            # namespace que raw après le renommage. On retire les canaux désélectionnés AVANT
            # load_data() pour ne lire sur disque que les canaux finalement gardés.
            present = [ch for ch in selected_channels if ch in raw.ch_names]
            if not present:
                with out_run:
                    print(f'[{file_id}] No selected channels found after renaming — skipped.')
                failed.append({'file_id': file_id, 'reason': 'no channels after rename'})
                progress.value += 1
                continue
            to_drop = [ch for ch in raw.ch_names if ch not in present]
            if to_drop:
                raw.drop_channels(to_drop)
            try:
                raw.load_data()   # ne lit sur disque que les canaux gardés
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Error reading EDF data: {e}')
                failed.append({'file_id': file_id, 'reason': f'EDF data loading: {e}'})
                progress.value += 1
                continue

            # [C] Resampling — fatal
            if do_resample and target_freq is not None:
                try:
                    raw.resample(target_freq, npad='auto', verbose=False)
                except Exception as e:
                    with out_run:
                        print(f'[{file_id}] Error during resampling: {e}')
                    failed.append({'file_id': file_id, 'reason': f'resampling: {e}'})
                    progress.value += 1
                    continue

            # [D] Re-référencement — non-fatal (warn + continuer sans re-ref)
            ref_channels = sub_config.get('ref_channels', [])
            try:
                if ref_channels == 'average':
                    raw.set_eeg_reference(ref_channels='average', verbose=False)
                elif isinstance(ref_channels, list) and ref_channels:
                    ref_present = [ch for ch in ref_channels if ch in raw.ch_names]
                    if ref_present:
                        raw.set_eeg_reference(ref_channels=ref_present, verbose=False)
                        raw.drop_channels(ref_present)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Error during re-referencing: {e} — skipped.')

            # [E] Filtrage — fatal
            if do_filter:
                try:
                    raw.filter(
                        l_freq=l_freq_val, h_freq=h_freq_val,
                        l_trans_bandwidth='auto', h_trans_bandwidth='auto',
                        filter_length='auto', method='fir', phase='zero-double',
                        fir_window='hamming', fir_design='firwin', verbose=False
                    )
                except Exception as e:
                    with out_run:
                        print(f'[{file_id}] Error during filtering: {e}')
                    failed.append({'file_id': file_id, 'reason': f'filtering: {e}'})
                    progress.value += 1
                    continue

            sf = raw.info['sfreq']

            _hypno_name = f'{file_id}{hypno_suffix.value}'
            hypno_path = hypno_lookup.get(os.path.normcase(_hypno_name), edf_path.parent / _hypno_name)
            if not hypno_path.exists():
                with out_run:
                    print(f'[{file_id}] Hypnogram not found: {hypno_path.name}')
                failed.append({'file_id': file_id, 'reason': f'hypno not found: {hypno_path.name}'})
                progress.value += 1
                continue
            try:
                expert_hypno = np.loadtxt(str(hypno_path), dtype=str).astype('<U10')
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Error loading hypnogram: {e}')
                failed.append({'file_id': file_id, 'reason': f'hypno loading: {e}'})
                progress.value += 1
                continue

            expected_epochs = int(np.floor(raw.n_times / sf / 30))
            if expected_epochs != len(expert_hypno):
                with out_run:
                    print(f'[{file_id}] Hypnogram/EEG length mismatch ({len(expert_hypno)} '
                          f'vs {expected_epochs} epochs) — skipped.')
                failed.append({'file_id': file_id, 'reason': 'hypno/EEG length mismatch'})
                progress.value += 1
                continue

            stage_mapping = {'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'R': 4}
            hypno_vec = np.full(len(expert_hypno), -1, dtype=float)
            for stage, value in stage_mapping.items():
                hypno_vec[expert_hypno == stage] = value

            # [F] Epoching — fatal
            try:
                epochs = mne.make_fixed_length_epochs(raw, duration=30, preload=True, verbose=False)
                epochs = epochs[:len(expert_hypno)]
                epochs.events[:, 2] = hypno_vec
                epochs.event_id     = stage_mapping
                n_epochs = len(epochs)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Error creating epochs: {e}')
                failed.append({'file_id': file_id, 'reason': f'epoching: {e}'})
                progress.value += 1
                continue

            if n_epochs == 0:
                with out_run:
                    print(f'[{file_id}] No epochs — skipped.')
                failed.append({'file_id': file_id, 'reason': 'no epochs'})
                progress.value += 1
                continue

            hypno_epochs   = expert_hypno[:n_epochs]
            epochs_data_uV = epochs.get_data() * 1e6  # (n_epochs, n_channels, n_times)

            # PSD — non-fatal (rejet 1/f désactivé si échec)
            try:
                n_per_seg = int(4 * sf)
                n_overlap = int(n_per_seg / 2)
                fmax_psd  = min(30.0, sf / 2 - 0.5)
                psds_obj  = epochs.compute_psd(
                    method='welch', fmin=0.5, fmax=fmax_psd,
                    n_fft=n_per_seg, n_overlap=n_overlap, n_per_seg=n_per_seg,
                    window='hann', verbose=False
                )
                psds_data_uV2 = psds_obj.get_data() * 1e12  # µV²/Hz
                psd_freqs     = psds_obj.freqs
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] PSD warning: {e} — 1/f rejection disabled.')
                psds_data_uV2 = None
                psd_freqs     = None

            # Event-based rejection mask (epoch-level) — non-fatal: a file with no event
            # companion just keeps the other 5 methods (never added to 'failed').
            event_epoch_mask = None
            if do_event:
                try:
                    events_df, ev_src = load_events(edf_path, csv_suffix.value)
                    if events_df is None:
                        with out_run:
                            print(f'[{file_id}] ⚠ No event companion found — event-based rejection skipped for this file.')
                    else:
                        event_epoch_mask = compute_event_epoch_mask(
                            events_df, event_remap_dict, event_types, n_epochs, 30.0
                        )
                except Exception as e:
                    with out_run:
                        print(f'[{file_id}] ⚠ Event rejection error: {e} — skipped for this file.')
                    event_epoch_mask = None

            # Masques de rejet — fatal
            try:
                mask_dict = compute_rejection_masks(
                    epochs_data_uV, sf, hypno_epochs,
                    ptp_thresh, flat_thresh_val, grad_thresh_val,
                    psd_freqs, psds_data_uV2, thresh_error_val, thresh_r2_val,
                    event_epoch_mask=event_epoch_mask
                )
                combined_matrix = build_combined_method_matrix(mask_dict)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Error computing rejection masks: {e}')
                failed.append({'file_id': file_id, 'reason': f'rejection masks: {e}'})
                progress.value += 1
                continue

            reject_epoch = np.zeros(n_epochs, dtype=bool)
            for name in mask_dict:
                reject_epoch |= mask_dict[name].any(axis=1)

            method_priority = [m for m in ['amplitude', 'gradient', 'flat', '1f_error', '1f_r2', 'event']
                               if m in mask_dict]
            reject_method   = np.full(n_epochs, 'none', dtype=object)
            for name in reversed(method_priority):
                reject_method[mask_dict[name].any(axis=1)] = name
            n_methods = sum(mask_dict[name].any(axis=1).astype(int) for name in method_priority)
            reject_method[n_methods > 1] = 'multiple'

            meta_rows = []
            for ei in range(n_epochs):
                row = {
                    'epoch_idx':     ei,
                    'stage':         hypno_epochs[ei],
                    'reject_flag':   bool(reject_epoch[ei]),
                    'reject_method': reject_method[ei],
                }
                for name in method_priority:
                    col = 'flag_' + name.replace('/', '').replace(' ', '')
                    row[col] = bool(mask_dict[name][ei].any())
                meta_rows.append(row)
            metadata_df = pd.DataFrame(meta_rows)

            # [G] Sauvegarde .fif — fatal (produit principal)
            try:
                epochs.metadata = metadata_df
                epochs.save(str(deriv_dir / f'{file_id}_all-epo.fif'), overwrite=True, verbose=False)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] Error saving .fif: {e}')
                failed.append({'file_id': file_id, 'reason': f'fif save: {e}'})
                progress.value += 1
                continue

            # [G] Sauvegarde epoch log TSV individuel — non-fatal
            try:
                epoch_log = build_epoch_rejection_log(mask_dict, hypno_epochs, file_id)
                epoch_log.to_csv(str(reports_dir / f'{file_id}_epoch_rejection.tsv'), sep='\t', index=False)
                all_epoch_logs.append(epoch_log)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Error saving epoch_rejection.tsv: {e}')

            # [G] Sauvegarde rejection summary TSV individuel — non-fatal
            rej_summary = None
            try:
                rej_summary = build_rejection_summary(mask_dict, hypno_epochs, file_id, epochs.ch_names, custom_stages=custom_stages)
                rej_summary.to_csv(str(reports_dir / f'{file_id}_rejection_summary.tsv'), sep='\t', index=False)
                all_rejection_summaries.append(rej_summary)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Error saving rejection_summary.tsv: {e}')

            n_rejected = int(reject_epoch.sum())
            pct_rej    = 100 * n_rejected / n_epochs if n_epochs > 0 else 0.0

            # [H] Rapport HTML + heatmap — non-fatal (.fif déjà sauvegardé)
            try:
                report = mne.Report(title=f'{file_id} — Preprocessing Phase 2', verbose=False)

                fig_hm = plot_rejection_heatmap(combined_matrix, epochs.ch_names, hypno_epochs, file_id, custom_stages=custom_stages)
                report.add_figure(fig=fig_hm, title='Rejection heatmap', section='Artifacts', image_format='PNG')
                plt.close(fig_hm)

                if rej_summary is not None:
                    rej_any      = rej_summary[rej_summary['method'] == 'any'][
                        ['stage', 'n_total', 'n_rejected', 'pct_rejected']]
                    summary_html = rej_any.to_html(index=False, border=0, justify='left')
                else:
                    summary_html = '<p><em>Per-stage summary unavailable.</em></p>'
                report.add_html(
                    html=(f'<p><b>Total epochs:</b> {n_epochs} | '
                          f'<b>Rejected:</b> {n_rejected} ({pct_rej:.1f}%)</p>'
                          + summary_html),
                    title='Rejection summary', section='Artifacts'
                )

                params_html = '<table style="border-collapse:collapse;font-size:.9em;">'
                if do_resample:
                    params_html += (f'<tr><td style="padding:3px 10px;"><b>Resampling</b></td>'
                                    f'<td style="padding:3px 10px;">{target_freq} Hz</td></tr>')
                else:
                    params_html += ('<tr><td style="padding:3px 10px;"><b>Resampling</b></td>'
                                    '<td style="padding:3px 10px;">no</td></tr>')
                if do_filter:
                    params_html += (f'<tr><td style="padding:3px 10px;"><b>Filter</b></td>'
                                    f'<td style="padding:3px 10px;">{l_freq_val}–{h_freq_val} Hz FIR zero-double Hamming</td></tr>')
                else:
                    params_html += ('<tr><td style="padding:3px 10px;"><b>Filter</b></td>'
                                    '<td style="padding:3px 10px;">no</td></tr>')
                params_html += (
                    f'<tr><td style="padding:3px 10px;"><b>Amplitude thresholds</b></td>'
                    f'<td style="padding:3px 10px;">W={ptp_thresh["W"]} N1={ptp_thresh["N1"]} '
                    f'N2={ptp_thresh["N2"]} N3={ptp_thresh["N3"]} R={ptp_thresh["R"]} µV</td></tr>'
                    f'<tr><td style="padding:3px 10px;"><b>Flat signal</b></td>'
                    f'<td style="padding:3px 10px;">&lt; {flat_thresh_val} µV</td></tr>'
                    f'<tr><td style="padding:3px 10px;"><b>Gradient</b></td>'
                    f'<td style="padding:3px 10px;">&gt; {grad_thresh_val} µV/sample</td></tr>'
                    f'<tr><td style="padding:3px 10px;"><b>1/f error</b></td>'
                    f'<td style="padding:3px 10px;">&gt; {thresh_error_val}</td></tr>'
                    f'<tr><td style="padding:3px 10px;"><b>1/f R²</b></td>'
                    f'<td style="padding:3px 10px;">&lt; {thresh_r2_val}</td></tr>'
                )
                if do_event:
                    _ev_state = 'applied' if event_epoch_mask is not None else 'no events found for this file'
                    params_html += (
                        f'<tr><td style="padding:3px 10px;"><b>Events</b></td>'
                        f'<td style="padding:3px 10px;">{_ev_state}: '
                        f'{", ".join(sorted(event_types))} '
                        f'(epoch containing event onset)</td></tr>'
                    )
                params_html += '</table>'
                report.save(str(reports_dir / f'{file_id}_preprocessing_report.html'),
                            overwrite=True, open_browser=False, verbose=False)
            except Exception as e:
                with out_run:
                    print(f'[{file_id}] ⚠ Error generating HTML report: {e}')

            edf_rel_str = str(edf_rel) if str(edf_rel) != '.' else ''
            with out_run:
                print(f'[{file_id}]  {n_rejected}/{n_epochs} epochs rejected ({pct_rej:.1f}%)'
                      + (f'  →  derivatives/{edf_rel_str}' if edf_rel_str else '  →  derivatives/'))

            progress.value = idx + 1

        except Exception as e:  # [A] catch-all par participant
            with out_run:
                print(f'[{file_id}] UNEXPECTED ERROR: {repr(e)}')
            failed.append({'file_id': file_id, 'reason': f'unexpected: {repr(e)}'})
            progress.value += 1

    # ---- Global outputs ---- [K]
    # Each global file is merged with existing data from previous runs.
    # Rows for attempted_ids are replaced (so a re-run on a participant replaces its old rows).
    # global_rejection_by_stage.tsv is regenerated from ALL individual _rejection_summary.tsv
    # files in reports_dir (mirrors quality_overview's dataset_overview.html strategy), so
    # it always reflects the complete state of the folder regardless of run history.

    # --- global_epoch_rejection.tsv ---
    global_epoch_path = reports_dir / 'global_epoch_rejection.tsv'
    try:
        if all_epoch_logs:
            df_new_epoch = pd.concat(all_epoch_logs, ignore_index=True)
            if global_epoch_path.exists() and attempted_ids:
                # dtype={'file_id': str}: same reason as quality_summary.tsv above.
                df_epoch_existing = pd.read_csv(str(global_epoch_path), sep='\t', dtype={'file_id': str})
                df_epoch_existing = df_epoch_existing[~df_epoch_existing['file_id'].astype(str).map(os.path.normcase).isin({os.path.normcase(s) for s in attempted_ids})]
                df_epoch_to_save = pd.concat([df_epoch_existing, df_new_epoch], ignore_index=True)
            else:
                df_epoch_to_save = df_new_epoch
            df_epoch_to_save.to_csv(str(global_epoch_path), sep='\t', index=False)
            with out_run:
                print('\nGlobal summary (per epoch)  →  global_epoch_rejection.tsv')
    except Exception as e:
        with out_run:
            print(f'⚠ Error saving global_epoch_rejection.tsv: {e}')

    # --- global_rejection_by_stage.tsv ---
    # Regenerated from all individual *_rejection_summary.tsv files present in reports_dir,
    # not just the current run — ensures the table always reflects the full dataset.
    global_stage_path = reports_dir / 'global_rejection_by_stage.tsv'
    try:
        all_summ_files = sorted(reports_dir.glob('*_rejection_summary.tsv'))
        if all_summ_files:
            all_summ_loaded = []
            for summ_path in all_summ_files:
                try:
                    all_summ_loaded.append(pd.read_csv(str(summ_path), sep='\t', dtype={'file_id': str}))
                except Exception as _e:
                    with out_run:
                        print(f'⚠ Could not load {summ_path.name}: {_e}')
            if all_summ_loaded:
                df_global_table = build_global_summary_table(all_summ_loaded, custom_stages=custom_stages)
                df_global_table.to_csv(str(global_stage_path), sep='\t', index=False)
                display_cols = ['stage'] + [c for c in df_global_table.columns if c.startswith('pct_')]
                pct_cols = [c for c in display_cols if c.startswith('pct_')]
                styled = (
                    df_global_table[display_cols].style
                    .set_table_styles([
                        {'selector': 'th', 'props': [('background', '#555'), ('color', '#fff'),
                                                      ('padding', '4px 12px'), ('text-align', 'left')]},
                        {'selector': 'td', 'props': [('padding', '3px 12px'),
                                                      ('border-bottom', '1px solid #eee')]},
                        {'selector': 'tr:nth-child(even) td', 'props': [('background', '#f8f8f8')]},
                    ])
                    .format({c: '{:.1f}' for c in pct_cols})
                    .hide(axis='index')
                )
                with out_run:
                    print('Global summary (per stage)  →  global_rejection_by_stage.tsv')
                    display(styled)
    except Exception as e:
        with out_run:
            print(f'⚠ Error generating global per-stage summary: {e}')

    # --- preprocessing_failed.tsv ---
    # Merge with existing: replace entries for attempted_ids (a participant that previously
    # failed and now succeeds is removed; one that fails again gets a fresh entry).
    failed_path = reports_dir / 'preprocessing_failed.tsv'
    if failed:
        try:
            df_failed_new = pd.DataFrame(failed)
            if failed_path.exists() and attempted_ids:
                df_failed_existing = pd.read_csv(str(failed_path), sep='\t', dtype={'file_id': str})
                df_failed_existing = df_failed_existing[~df_failed_existing['file_id'].astype(str).map(os.path.normcase).isin({os.path.normcase(s) for s in attempted_ids})]
                df_failed_merged = pd.concat([df_failed_existing, df_failed_new], ignore_index=True)
            else:
                df_failed_merged = df_failed_new
            df_failed_merged.to_csv(str(failed_path), sep='\t', index=False)
            with out_run:
                print(f'\n{len(failed)} participant(s) failed  →  preprocessing_failed.tsv')
        except Exception as e:
            with out_run:
                print(f'\n{len(failed)} participant(s) failed. '
                      f'⚠ Error saving preprocessing_failed.tsv: {e}')
    elif failed_path.exists() and attempted_ids:
        # All attempted participants succeeded: clean up any stale entries in failed.tsv.
        try:
            df_failed_existing = pd.read_csv(str(failed_path), sep='\t', dtype={'file_id': str})
            df_failed_cleaned = df_failed_existing[~df_failed_existing['file_id'].astype(str).map(os.path.normcase).isin({os.path.normcase(s) for s in attempted_ids})]
            if df_failed_cleaned.empty:
                failed_path.unlink()
            elif len(df_failed_cleaned) < len(df_failed_existing):
                df_failed_cleaned.to_csv(str(failed_path), sep='\t', index=False)
        except Exception as e:
            with out_run:
                print(f'⚠ Error updating preprocessing_failed.tsv: {e}')

    elapsed = time.time() - t_start
    mins, secs = divmod(elapsed, 60)
    elapsed_str = f'{int(mins)} min {secs:.0f} s' if mins >= 1 else f'{secs:.1f} s'

    n_this_run = len(attempted_ids) - len(failed)
    n_total_in_folder = len(list(reports_dir.glob('*_preprocessing_report.html')))
    with out_run:
        print(f'\n=== Run complete ({elapsed_str}) ===')
        print(f'Processed this run    : {n_this_run}')
        if failed:
            print(f'Failed this run       : {len(failed)}')
        print(f'Total in folder       : {n_total_in_folder}')

    progress_lbl.value = f'Done. ({elapsed_str})'
    btn.disabled = False


btn_run.on_click(run_preprocessing)

display(
    widgets.HBox([btn_load, load_info]),
    ctrl_row,
    vbox_participants,
    HTML('<hr style="margin:16px 0;">'),
    btn_run,
    widgets.HBox([progress, progress_lbl]),
    out_run,
)